In [ ]:
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

In [ ]:

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [20]:
# Let's try out this utility

ed = fetch_website_contents("https://edwarddonner.com")
print(ed)

Home - Edward Donner

Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.io
. I was previously founder and CEO of AI startup untapt,
acquired in 2021
, and a Managing Director at JPMorgan.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 900,000 enrollments across 194 countries. The
full curriculum is here
. If you’re visiting from one of my courses – I’m super grateful!
F

## Types of prompts

You may know this already - but if not, you will get very familiar with it!

Models like GPT have been trained to receive instructions in a particular way.

They expect to receive:

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [21]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [22]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

## Messages

The API from OpenAI expects to receive messages in a particular structure.
Many of the other APIs share this structure:

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```
To give you a preview, the next 2 cells make a rather simple call - we won't stretch the mighty GPT (yet!)

## And now let's build useful messages for GPT-4.1-mini, using a function

In [24]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [25]:
# Try this out, and then try for a few more websites

messages_for(ed)

[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nHome - Edward Donner\n\nSkip to content\nAvatar\nCurriculum\nProficiency\nC4\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of AI startup\nNebula

## Time to bring it together - the API for OpenAI is very simple!

In [ ]:
# And now: call ollama running locally on the laptop.

def summarize(url):
    website = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model = "llama3.2",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [27]:
summarize("https://edwarddonner.com")

"### A Website Primarily About AI and Networking by Ed Donner\n\nEd Donner is a co-founder of AI startup Nebula.io, which uses Large Language Models (LLMs) for various purposes. He has developed several popular Udemy courses on the subject, which have attracted a global following of nearly 900,000 students across 194 countries. The website mainly contains introductory information about Ed, his background, and his work on LLMs, as well as announcements of AI-related resources and posts.\n\n### Notable Announcements:\n\n- January 4, 2026: AI Builder with n8n - Create Agents and Voice Agents - RESOURCES\n- February 17, 2026: AI Coder: Vibe Coder to Agentic Engineer – RESOURCES\n- September 15, 2025: AI Engineering MLOps Track – Deploy AI to Production – RESOURCES\n- May 28, 2025: Guidance on taking Ed's AI courses"

In [28]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [30]:
display_summary("https://aws.com")

A typical corporate website...

AWS seems to be trying to convince you that it's the ultimate cloud solution, because, why not? They offer a wide range of services for almost anything, from AI and machine learning to computing and analytics. There's even a thing called "re:Invent 2026" which sounds like a real party.

The news section appears to be just a blog, with some recent posts mentioning the launch of Amazon Quick, a tool for automating business tasks, and Amazon Bedrock, a platform for building generative AI apps. They also talk about their trust center, which probably involves a lot of paperwork.

The website is a giant, well-organized mess of features and solutions, but at the end of the day, it's just a bunch of techy stuff to help you save time and money. Because who doesn't love doing more with less?

# Let's try more websites

Note that this will only work on websites that can be scraped using this simplistic approach.

Websites that are rendered with Javascript, like React apps, won't show up. See the community-contributions folder for a Selenium implementation that gets around this. You'll need to read up on installing Selenium (ask ChatGPT!)

Also Websites protected with CloudFront (and similar) may give 403 errors - many thanks Andy J for pointing this out.

But many websites will work just fine!

In [31]:
display_summary("https://lambda.ai")

**Summary:** This website appears to be a platform for accessing and utilizing artificial intelligence (AI) computing resources, specifically focused on cloud-based services called Lambda. It offers various features such as supercomputing, superclusters, 1-Click Clusters, and access to high-end GPUs.

**News/Announcements:** The website includes a log-in section, a session ID, and an agent handshake that appears to be a part of a larger system. There's also a list of actions, such as launching a GPU instance, with a "Primary CTA detected --" note. No clear news or announcements are mentioned.

In [32]:
display_summary("https://anthropic.com")

**AI-Promises and Responsibilities**
Anthropic aims to reap AI's benefits while avoiding its risks. They're tackling tough questions on safety, governance, and societal impacts. They've recently released Opus 5, a major upgrade with improved coding, agent capacity, and professionalism.